In [12]:
# from roboflow import Roboflow
# API_KEY = None
# rf = Roboflow(api_key=API_KEY)
# project = rf.workspace("body-fluid-qgrh0").project("white-blood-cell-ljwxj")
# version = project.version(1)
# dataset = version.download("yolov11")

In [13]:
from ultralytics import YOLO

model = YOLO("yolo11m.yaml").load("yolo11m.pt")  # build from YAML and transfer weights

Transferred 649/649 items from pretrained weights


In [14]:
# Train the model
# results = model.train(data="white_blood/data.yaml", epochs=50, imgsz=640)
results = model.train(data="yolo_dataset/data.yaml", epochs=50, imgsz=300)

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA A10, 22599MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=300, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train21, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=

In [15]:
import cv2
import numpy as np

def crop_and_label_objects(result):
    """
    Extracts bounding boxes from a YOLO result, crops them from the clean image,
    and grabs the associated class labels.
    
    Args:
        result: A single Ultralytics Results object.
        
    Returns:
        list: A list of dictionaries, each containing the cropped image array, 
              the class ID (int), and the class name (string).
    """
    extracted_data = []
    
    # Get the clean original image and convert to RGB
    orig_img_rgb = cv2.cvtColor(result.orig_img, cv2.COLOR_BGR2RGB)
    
    # Get the dictionary mapping class IDs to class names (e.g., {0: 'cell_x', 1: 'cell_y'})
    class_names = result.names
    
    for box in result.boxes:
        # 1. Get coordinates
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        
        # 2. Get the label
        # box.cls[0] is a tensor, so we use .item() to get the raw Python number, then cast to int
        class_id = int(box.cls[0].item()) 
        class_name = class_names[class_id]
        
        # 3. Crop the image
        crop = orig_img_rgb[y1:y2, x1:x2]
        
        # 4. Store them together if the crop is valid
        if crop.size > 0:
            extracted_data.append({
                "image": crop,
                "class_id": class_id,
                "class_name": class_name
            })
            
    return extracted_data

In [19]:
from PIL import Image

# Run prediction
results = model.predict(source="white_blood/test/images", save=True, conf=0.25)

# Process the first 3 results
for i, r in enumerate(results[:20]):

    
    # Call our updated function
    detected_objects = crop_and_label_objects(r)
    
    print(f"\n--- Image {i+1}: Found {len(detected_objects)} objects ---")
    
    for j, obj_data in enumerate(detected_objects):
        
        # Unpack the dictionary
        crop_array = obj_data["image"]
        label_id = obj_data["class_id"]
        label_name = obj_data["class_name"]
        
        
        print(f"Object {j+1}: It's a '{label_name}' (Class ID: {label_id})")
        
        # Show the image (optional)
        im = Image.fromarray(crop_array)
        # im.show() # Uncomment to pop open the images
        
        # ---> Pass `crop_array` AND `label_id` to your CNN here <---


image 1/500 /workspace/white_blood/test/images/BA_266178_jpg.rf.b4891d81cfba72ccab9f9b93ff81069b.jpg: 320x320 1 lymphocyte, 18.2ms
image 2/500 /workspace/white_blood/test/images/BA_27002_jpg.rf.dbd5fed738ff1fbbacbd3bdb18a4d4f1.jpg: 320x320 1 lymphocyte, 16.1ms
image 3/500 /workspace/white_blood/test/images/BA_274864_jpg.rf.a121cd9c14e2409e35aeefb2532986bf.jpg: 320x320 (no detections), 13.9ms
image 4/500 /workspace/white_blood/test/images/BA_281662_jpg.rf.76dc9d2af2b4a138a6a8a2e0c6d48b63.jpg: 320x320 1 eosinophil, 7.2ms
image 5/500 /workspace/white_blood/test/images/BA_307830_jpg.rf.44c98d0346e52259f7c965cbc1a8fb93.jpg: 320x320 1 eosinophil, 7.2ms
image 6/500 /workspace/white_blood/test/images/BA_320878_jpg.rf.aec02465af9fd811674b1630f5e0e85c.jpg: 320x320 1 lymphocyte, 7.1ms
image 7/500 /workspace/white_blood/test/images/BA_328970_jpg.rf.d19dcb2eefb4b4835f96059a27bf7089.jpg: 320x320 1 lymphocyte, 7.1ms
image 8/500 /workspace/white_blood/test/images/BA_35932_jpg.rf.d8e2a7216c2b2df08eb86

In [20]:
# Run validation
# YOLO calculates IoU internally to produce these results
metrics = model.val(data="/workspace/white-blood-cell-1/data.yaml", conf=0.25)

# The 'Fitness' metric is actually a weighted average of IoU-based precision:
# Fitness = 0.1 * mAP@50 + 0.9 * mAP@50-95
print(f"Model Fitness (IoU-based score): {metrics.fitness:.4f}")

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA A10, 22599MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1055.9±135.2 MB/s, size: 27.4 KB)
val: Scanning /workspace/white-blood-cell-1/valid/labels.cache... 1000 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1000/1000 524.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 63/63 14.7it/s 4.3s0.1s
                   all       1000       1009     0.0286     0.0609     0.0173     0.0137
        seg_neutrophil        199        199      0.114      0.246     0.0711     0.0567
            lymphocyte        200        207     0.0235     0.0483     0.0123    0.00957
              monocyte        200        201          0          0          0          0
            eosinophil        200        202     0.0058     0.0099    0.00293    0.00219
              basophil        200        200          0          0          0          0
Speed: 0.2ms pre

In [21]:
metrics = model.val(data="/workspace/yolo_dataset/data.yaml", conf=0.25)

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA A10, 22599MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 141.0±17.5 MB/s, size: 6.5 KB)
val: Scanning /workspace/yolo_dataset/valid/labels.cache... 295 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 295/295 77.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 13.2it/s 1.4s0.1s
                   all        295        295      0.646      0.974      0.797      0.651
        seg_neutrophil         92         92      0.652      0.957      0.879      0.733
            lymphocyte         76         76      0.784      0.953      0.916      0.775
              monocyte         54         54      0.841      0.981      0.968      0.758
            eosinophil         55         55      0.578      0.982      0.664      0.521
              basophil         18         18      0.375          1      0.557      0.469
Speed: 0.3ms preprocess, 2.5m